# Main Grad-CAM Panel Figure

This first section creates the manuscript-style Grad-CAM panel figure. It uses the selected 3D ResNet model at epoch 20, ranks candidate slices by how well the Grad-CAM heatmap overlaps the tumor region, saves every small image separately, and saves one 2 x N panel for label=1 and one 2 x N panel for label=0.

In [1]:

# ============================================================
# Section 0. Main Grad-CAM panel figure
# ============================================================

import os
import sys
from pathlib import Path

sys.path.append('/host/d/Github/')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

import Osteosarcoma.Image_3D.Generator_ResNet as Generator_ResNet
import Osteosarcoma.Image_3D.resnet.model as resnet_model

# ------------------------------
# User-facing settings
# ------------------------------
N_EXAMPLES_PER_LABEL = 5
EPOCH_FOR_MAIN_GRADCAM = 20

LABEL = 'Prognosis'
TRIAL_NAME = 'resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam'
MODEL_SETTING = 'random0_all_fold56'
MODEL_DEPTH = 18
IN_CHANNELS = 3
TARGET_SIZE = (96, 96, 64)

# Representative-slice eligibility rule requested by the user:
# the displayed tumor mask must be fully contained within the 96 x 96 image view.
# A slice is rejected if tumor pixels touch the image boundary, because that suggests
# the tumor may have been cropped by the fixed-size ROI view.
TUMOR_VIEW_MARGIN_PIXELS = 2
MAX_TUMOR_VIEW_FRACTION = 0.75

DATA_ROOT = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'

MODEL_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/models/model-{EPOCH_FOR_MAIN_GRADCAM}.pt'
PREDICTION_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/predictions/prediction_eval_epoch{EPOCH_FOR_MAIN_GRADCAM}_fold0123456.xlsx'
METRICS_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/predictions/metrics_eval_epoch{EPOCH_FOR_MAIN_GRADCAM}_fold0123456.xlsx'

RESULTS_DIR = Path('/host/d/projects/Habitats/results')
GRADCAM_OUT_DIR = RESULTS_DIR / 'grad-cam'
GRADCAM_OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# For visual consistency with the manuscript figures.
font_candidates = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf',
]
for font_path in font_candidates:
    if os.path.exists(font_path):
        font_manager.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'Times New Roman'
        break

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

def save_pdf_and_ppt_safe_svg(fig, pdf_path, **kwargs):
    """Save the normal PDF plus a PPT-friendly SVG copy.

    The SVG suffix is `_ppt_safe.svg`, so original PDF manuscript outputs stay unchanged.
    """
    pdf_path = Path(pdf_path)
    fig.savefig(pdf_path, **kwargs)
    svg_path = pdf_path.with_name(pdf_path.stem + '_ppt_safe.svg')
    fig.savefig(svg_path, format='svg', **kwargs)
    return svg_path

print('Device:', DEVICE)
print('Model:', MODEL_PATH)
print('Prediction:', PREDICTION_PATH)
print('Output folder:', GRADCAM_OUT_DIR)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(MODEL_PATH)
if not os.path.exists(PREDICTION_PATH):
    raise FileNotFoundError(
        f'Prediction file not found: {PREDICTION_PATH}\n'
        'Run Image_3D/resnet/predict.py for this epoch before executing this section.'
    )


class GradCAM3DPanel:
    def __init__(self, model, target_layer):
        self.activations = []
        self.gradients = []
        self.handles = [
            target_layer.register_forward_hook(self._forward_hook),
            target_layer.register_full_backward_hook(self._backward_hook),
        ]

    def _forward_hook(self, module, inputs, output):
        self.activations.append(output)

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients.append(grad_output[0])

    def clear(self):
        self.activations = []
        self.gradients = []

    def close(self):
        for handle in self.handles:
            handle.remove()

    def compute(self, target_size):
        if len(self.activations) == 0 or len(self.gradients) == 0:
            raise RuntimeError('No activations/gradients captured. Did backward run?')
        activation = self.activations[-1]
        gradient = self.gradients[-1]
        weights = gradient.mean(dim=(2, 3, 4), keepdim=True)
        cam = F.relu((weights * activation).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=target_size, mode='trilinear', align_corners=False)
        cam = cam[0, 0].detach().cpu().numpy().astype(np.float32)
        cam_min = float(np.min(cam))
        cam_max = float(np.max(cam))
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam, dtype=np.float32)
        return cam


def load_resnet_checkpoint(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            key = key[len('module.'):]
        cleaned[key] = value
    model.load_state_dict(cleaned, strict=True)
    return model


def build_model_for_gradcam():
    model = resnet_model.build_resnet3d_model(
        model_depth=MODEL_DEPTH,
        num_classes=2,
        in_channels=IN_CHANNELS,
    )
    model = load_resnet_checkpoint(model, MODEL_PATH, DEVICE)
    model.to(DEVICE)
    model.eval()
    return model


def build_single_case_dataset(patient_set, patient_index, label):
    x_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'img.nii.gz')
    return Generator_ResNet.Dataset_3D(
        [patient_set],
        [str(patient_index)],
        [x_file],
        [int(label)],
        DATA_ROOT,
        target_image_size=TARGET_SIZE,
        normalize_factor='medicalnet',
        only_tumor_pixels='seg',
        augment_context='full',
        shuffle=False,
        augment=False,
        augment_frequency=0,
    )


def orient_slice(slice_2d):
    # Keep the same display orientation used in the existing Grad-CAM figures.
    return np.flipud(np.rot90(slice_2d, k=1))


def normalize_display(img):
    img = img.astype(np.float32)
    valid = img[img != 0]
    if valid.size > 20:
        lo, hi = np.percentile(valid, [1, 99])
    else:
        lo, hi = float(np.min(img)), float(np.max(img))
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((img - lo) / (hi - lo), 0, 1)


def overlay_mask_on_gray(gray, mask, color=(1.0, 0.0, 0.0), alpha=0.42):
    rgb = np.repeat(gray[..., None], 3, axis=-1)
    color_arr = np.asarray(color, dtype=np.float32)
    mask = mask.astype(bool)
    rgb[mask] = (1 - alpha) * rgb[mask] + alpha * color_arr
    return np.clip(rgb, 0, 1)


def make_cam_overlay(gray, cam, alpha=0.48):
    heat_rgb = plt.get_cmap('jet')(cam)[..., :3]
    gray_rgb = np.repeat(gray[..., None], 3, axis=-1)
    return np.clip((1 - alpha) * gray_rgb + alpha * heat_rgb, 0, 1)


def normalize_cam_slice_for_display(cam_slice):
    """Normalize the selected 2D Grad-CAM slice to [0, 1] for visualization.

    The 3D CAM volume is still used for case/slice ranking. For the final
    manuscript display, many Grad-CAM figures normalize each shown 2D map
    independently so label=0 and label=1 cases are visually comparable.
    """
    cam_slice = cam_slice.astype(np.float32)
    cam_min = float(np.min(cam_slice))
    cam_max = float(np.max(cam_slice))
    if cam_max <= cam_min:
        return np.zeros_like(cam_slice, dtype=np.float32)
    return ((cam_slice - cam_min) / (cam_max - cam_min)).astype(np.float32)


def save_image_only(img, output_path):
    output_path = Path(output_path)
    fig = plt.figure(figsize=(2.1, 2.1), dpi=300)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img, vmin=0, vmax=1)
    ax.set_axis_off()
    save_pdf_and_ppt_safe_svg(fig, output_path.with_suffix('.pdf'), bbox_inches='tight', pad_inches=0)
    fig.savefig(output_path.with_suffix('.png'), format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)


def get_case_raw_and_tensor(patient_set, patient_index, label):
    dataset = build_single_case_dataset(patient_set, patient_index, label)
    img_file = dataset.x_file_list[0]
    label_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'label.nii.gz')
    bbox_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'bbox_mask.nii.gz')
    raw_stack = dataset.load_file(img_file, label_file, bbox_file)
    x_tensor, y_tensor = dataset[0]
    return raw_stack, x_tensor.unsqueeze(0), int(y_tensor.item())


def tumor_slice_view_stats(tumor_mask_slice):
    tumor_mask_slice = np.asarray(tumor_mask_slice).astype(bool)
    h, w = tumor_mask_slice.shape
    area = int(tumor_mask_slice.sum())
    stats = {
        'tumor_area': area,
        'tumor_area_fraction': float(area / (h * w)),
        'tumor_bbox_y_min': np.nan,
        'tumor_bbox_y_max': np.nan,
        'tumor_bbox_x_min': np.nan,
        'tumor_bbox_x_max': np.nan,
        'tumor_touches_view_border': True,
        'tumor_fully_in_view': False,
    }
    if area <= 0:
        return stats

    yy, xx = np.where(tumor_mask_slice)
    y_min, y_max = int(yy.min()), int(yy.max())
    x_min, x_max = int(xx.min()), int(xx.max())

    margin = int(TUMOR_VIEW_MARGIN_PIXELS)
    touches_border = (
        y_min <= margin
        or x_min <= margin
        or y_max >= (h - 1 - margin)
        or x_max >= (w - 1 - margin)
    )
    area_too_large = stats['tumor_area_fraction'] >= float(MAX_TUMOR_VIEW_FRACTION)

    stats.update({
        'tumor_bbox_y_min': y_min,
        'tumor_bbox_y_max': y_max,
        'tumor_bbox_x_min': x_min,
        'tumor_bbox_x_max': x_max,
        'tumor_touches_view_border': bool(touches_border),
        'tumor_fully_in_view': bool((not touches_border) and (not area_too_large)),
    })
    return stats


def select_display_slice(cam_volume, tumor_mask_volume):
    tumor_area = tumor_mask_volume.sum(axis=(0, 1))
    cam_tumor_overlap = (cam_volume * tumor_mask_volume).sum(axis=(0, 1))
    max_area = float(tumor_area.max())
    if max_area <= 0:
        raise RuntimeError('No tumor pixels found in ROI view.')

    # Candidate slices must contain meaningful tumor area and must show the whole
    # tumor mask inside the image view. This is checked slice by slice because a
    # large tumor may be cropped on one slice but fully visible on another.
    min_area = max(5, 0.10 * max_area)
    valid_z = []
    for z in np.where(tumor_area >= min_area)[0]:
        stats = tumor_slice_view_stats(tumor_mask_volume[:, :, int(z)])
        if stats['tumor_fully_in_view']:
            valid_z.append(int(z))

    if len(valid_z) == 0:
        raise RuntimeError(
            'No eligible display slice with tumor mask fully inside the 96x96 view. '
            'This case is skipped for representative Grad-CAM selection.'
        )

    valid_z = np.asarray(valid_z, dtype=int)
    return int(valid_z[np.argmax(cam_tumor_overlap[valid_z])])


def run_gradcam_case(model, gradcam, patient_set, patient_index, true_label, target_class):
    raw_stack, x_tensor, y_true = get_case_raw_and_tensor(patient_set, patient_index, true_label)
    x_tensor = x_tensor.to(DEVICE, dtype=torch.float32)

    model.eval()
    model.zero_grad(set_to_none=True)
    gradcam.clear()

    logits = model(x_tensor)
    prob_class1 = torch.softmax(logits, dim=1)[0, 1].detach().cpu().item()
    pred_class = int(torch.argmax(logits, dim=1).detach().cpu().item())
    logits[0, int(target_class)].backward()
    cam_volume = gradcam.compute(TARGET_SIZE)

    # Channel meaning from Generator_ResNet: [full context, bbox-only, tumor-only].
    tumor_mask_volume = raw_stack[2] != 0
    bbox_mask_volume = raw_stack[1] != 0
    if not np.any(tumor_mask_volume):
        tumor_mask_volume = bbox_mask_volume.copy()

    total_cam = float(cam_volume.sum()) + 1e-12
    tumor_cam_fraction = float(cam_volume[tumor_mask_volume].sum() / total_cam) if np.any(tumor_mask_volume) else np.nan
    bbox_cam_fraction = float(cam_volume[bbox_mask_volume].sum() / total_cam) if np.any(bbox_mask_volume) else np.nan

    z_index = select_display_slice(cam_volume, tumor_mask_volume)
    mri_slice = orient_slice(raw_stack[0, :, :, z_index])
    tumor_mask_slice = orient_slice(tumor_mask_volume[:, :, z_index])
    cam_slice_raw = orient_slice(cam_volume[:, :, z_index])
    # Display-only normalization: normalize the selected 2D slice independently.
    # This keeps the 3D CAM calculation/ranking unchanged, but makes label=0 and
    # label=1 example panels visually comparable in the same way many Grad-CAM
    # paper figures are rendered.
    cam_slice = normalize_cam_slice_for_display(cam_slice_raw)
    gray = normalize_display(mri_slice)

    selected_slice_view_stats = tumor_slice_view_stats(tumor_mask_slice)
    if not selected_slice_view_stats['tumor_fully_in_view']:
        raise RuntimeError(
            f'Selected slice z={z_index} is not fully in view: {selected_slice_view_stats}'
        )

    tumor_overlay = overlay_mask_on_gray(gray, tumor_mask_slice, color=(1.0, 0.0, 0.0), alpha=0.42)
    cam_overlay = make_cam_overlay(gray, cam_slice, alpha=0.48)

    # Slice-level localization score: how much of the hottest 20% CAM area is in tumor.
    cam_threshold = float(np.quantile(cam_slice, 0.80))
    hot_mask = cam_slice >= cam_threshold
    hot_in_tumor_fraction = float((hot_mask & tumor_mask_slice).sum() / max(int(hot_mask.sum()), 1))
    tumor_cam_mean = float(cam_slice[tumor_mask_slice].mean()) if np.any(tumor_mask_slice) else np.nan
    background_cam_mean = float(cam_slice[~tumor_mask_slice].mean()) if np.any(~tumor_mask_slice) else np.nan
    tumor_to_background_cam_ratio = float(tumor_cam_mean / (background_cam_mean + 1e-12)) if np.isfinite(tumor_cam_mean) else np.nan

    target_probability = prob_class1 if int(true_label) == 1 else 1.0 - prob_class1
    localization_score = (
        2.0 * hot_in_tumor_fraction
        + 1.0 * tumor_cam_fraction
        + 0.2 * min(max(tumor_to_background_cam_ratio, 0), 5)
        + 0.1 * target_probability
    )

    return {
        'patient_set': str(patient_set),
        'patient_index': str(patient_index),
        'true_label': int(true_label),
        'target_class': int(target_class),
        'probability_class1': float(prob_class1),
        'prediction_class_argmax': pred_class,
        'target_probability': float(target_probability),
        'display_z_index': int(z_index),
        'tumor_area_on_display_slice': int(tumor_mask_slice.sum()),
        'tumor_area_fraction_on_display_slice': float(selected_slice_view_stats['tumor_area_fraction']),
        'tumor_fully_in_view_on_display_slice': bool(selected_slice_view_stats['tumor_fully_in_view']),
        'tumor_touches_view_border_on_display_slice': bool(selected_slice_view_stats['tumor_touches_view_border']),
        'tumor_bbox_y_min_on_display_slice': selected_slice_view_stats['tumor_bbox_y_min'],
        'tumor_bbox_y_max_on_display_slice': selected_slice_view_stats['tumor_bbox_y_max'],
        'tumor_bbox_x_min_on_display_slice': selected_slice_view_stats['tumor_bbox_x_min'],
        'tumor_bbox_x_max_on_display_slice': selected_slice_view_stats['tumor_bbox_x_max'],
        'tumor_cam_fraction_volume': tumor_cam_fraction,
        'bbox_cam_fraction_volume': bbox_cam_fraction,
        'hot_in_tumor_fraction_slice': hot_in_tumor_fraction,
        'tumor_cam_mean_slice': tumor_cam_mean,
        'background_cam_mean_slice': background_cam_mean,
        'tumor_to_background_cam_ratio_slice': tumor_to_background_cam_ratio,
        'localization_score': float(localization_score),
        'tumor_overlay': tumor_overlay,
        'cam_overlay': cam_overlay,
    }


def make_label_panel(selected_results, label_value, output_stem):
    n = len(selected_results)
    fig = plt.figure(figsize=(2.05 * n, 4.1), dpi=300)
    gs = fig.add_gridspec(
        2,
        n,
        left=0.0,
        right=1.0,
        top=1.0,
        bottom=0.0,
        wspace=0.015,
        hspace=0.015,
    )
    for col, result in enumerate(selected_results):
        ax_top = fig.add_subplot(gs[0, col])
        ax_bottom = fig.add_subplot(gs[1, col])
        ax_top.imshow(result['tumor_overlay'], vmin=0, vmax=1)
        ax_bottom.imshow(result['cam_overlay'], vmin=0, vmax=1)
        ax_top.set_axis_off()
        ax_bottom.set_axis_off()
    pdf_path = GRADCAM_OUT_DIR / f'{output_stem}.pdf'
    png_path = GRADCAM_OUT_DIR / f'{output_stem}.png'
    save_pdf_and_ppt_safe_svg(fig, pdf_path, bbox_inches='tight', pad_inches=0)
    fig.savefig(png_path, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    return pdf_path, png_path


def save_selected_small_images(selected_results, label_value):
    saved_rows = []
    label_name = f'label{label_value}'
    for i, result in enumerate(selected_results, start=1):
        case_id = f"{label_name}_example{i:02d}_{result['patient_set']}_{result['patient_index']}_z{result['display_z_index']:02d}"
        tumor_stem = GRADCAM_OUT_DIR / f'{case_id}_tumor_overlay'
        cam_stem = GRADCAM_OUT_DIR / f'{case_id}_gradcam_overlay'
        save_image_only(result['tumor_overlay'], tumor_stem)
        save_image_only(result['cam_overlay'], cam_stem)
        saved_rows.append({
            **{k: v for k, v in result.items() if k not in ['tumor_overlay', 'cam_overlay']},
            'example_order': i,
            'label_group': label_value,
            'tumor_overlay_pdf': str(tumor_stem.with_suffix('.pdf')),
            'tumor_overlay_png': str(tumor_stem.with_suffix('.png')),
            'gradcam_overlay_pdf': str(cam_stem.with_suffix('.pdf')),
            'gradcam_overlay_png': str(cam_stem.with_suffix('.png')),
        })
    return saved_rows


prediction_df = pd.read_excel(PREDICTION_PATH)
metrics_df = pd.read_excel(METRICS_PATH)
best_threshold = float(metrics_df['Best_threshold'].iloc[0]) if 'Best_threshold' in metrics_df.columns else 0.5
print('Epoch:', EPOCH_FOR_MAIN_GRADCAM)
print('Best threshold:', best_threshold)
print('Prediction table shape:', prediction_df.shape)

model = build_model_for_gradcam()
gradcam = GradCAM3DPanel(model, model.layer4[-1])

all_candidate_rows = []
selected_by_label = {}

for label_value in [1, 0]:
    print('\n============================================================')
    print('Scanning Grad-CAM candidates for label =', label_value)
    subset = prediction_df[prediction_df['label'].astype(int) == label_value].copy()
    # Prefer correctly classified cases, but keep all cases as fallback.
    if label_value == 1:
        subset['is_correct_by_threshold'] = subset['probability'] >= best_threshold
        subset['target_probability'] = subset['probability']
    else:
        subset['is_correct_by_threshold'] = subset['probability'] < best_threshold
        subset['target_probability'] = 1.0 - subset['probability']
    subset = subset.sort_values(['is_correct_by_threshold', 'target_probability'], ascending=[False, False])

    label_results = []
    for row_i, row in subset.iterrows():
        patient_set = str(row['Patient_set'])
        patient_index = str(row['Patient_index'])
        try:
            result = run_gradcam_case(
                model,
                gradcam,
                patient_set=patient_set,
                patient_index=patient_index,
                true_label=int(label_value),
                target_class=int(label_value),
            )
            result['fold'] = int(row['fold']) if 'fold' in row else np.nan
            result['is_correct_by_threshold'] = bool(row['is_correct_by_threshold'])
            label_results.append(result)
        except Exception as exc:
            print('  Failed:', patient_set, patient_index, repr(exc))
        if len(label_results) % 25 == 0 and len(label_results) > 0:
            print('  scanned', len(label_results), 'cases for label', label_value)

    # Rank by localization first, then target probability. This selects examples where
    # the heatmap is concentrated within the tumor and the model output is sensible.
    candidate_df = pd.DataFrame([
        {k: v for k, v in r.items() if k not in ['tumor_overlay', 'cam_overlay']}
        for r in label_results
    ])
    candidate_df['label_group'] = label_value
    candidate_df = candidate_df.sort_values(
        ['is_correct_by_threshold', 'localization_score', 'hot_in_tumor_fraction_slice', 'tumor_cam_fraction_volume', 'target_probability'],
        ascending=[False, False, False, False, False],
    )
    all_candidate_rows.append(candidate_df)

    selectable_df = candidate_df[candidate_df['tumor_fully_in_view_on_display_slice'].astype(bool)].copy()
    print(
        'Display-quality eligible cases for label',
        label_value,
        ':',
        selectable_df.shape[0],
        '/',
        candidate_df.shape[0],
    )
    if selectable_df.shape[0] < N_EXAMPLES_PER_LABEL:
        raise RuntimeError(
            f'Only {selectable_df.shape[0]} display-quality eligible cases found for label {label_value}; '
            f'need {N_EXAMPLES_PER_LABEL}. Consider relaxing TUMOR_VIEW_MARGIN_PIXELS '
            'or MAX_TUMOR_VIEW_FRACTION.'
        )

    selected = []
    used_cases = set()
    result_lookup = {(r['patient_set'], r['patient_index']): r for r in label_results}
    for _, cand in selectable_df.iterrows():
        key = (str(cand['patient_set']), str(cand['patient_index']))
        if key in used_cases:
            continue
        selected.append(result_lookup[key])
        used_cases.add(key)
        if len(selected) >= N_EXAMPLES_PER_LABEL:
            break

    if len(selected) < N_EXAMPLES_PER_LABEL:
        raise RuntimeError(
            f'Only {len(selected)} unique display-quality cases selected for label {label_value}; '
            f'need {N_EXAMPLES_PER_LABEL}.'
        )

    selected_by_label[label_value] = selected
    print('Selected cases for label', label_value)
    for i, r in enumerate(selected, start=1):
        print(
            f"  {i}. {r['patient_set']}/{r['patient_index']} z={r['display_z_index']} "
            f"prob={r['probability_class1']:.3f} hot_in_tumor={r['hot_in_tumor_fraction_slice']:.3f} "
            f"tumor_cam={r['tumor_cam_fraction_volume']:.3f} area_frac={r['tumor_area_fraction_on_display_slice']:.3f} "
            f"touch_border={r['tumor_touches_view_border_on_display_slice']} score={r['localization_score']:.3f}"
        )

gradcam.close()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

candidate_all_df = pd.concat(all_candidate_rows, ignore_index=True)
candidate_path = GRADCAM_OUT_DIR / 'gradcam_epoch20_candidate_ranking.xlsx'
candidate_all_df.to_excel(candidate_path, index=False)
print('\nSaved candidate ranking:', candidate_path)

selected_rows = []
for label_value, selected_results in selected_by_label.items():
    selected_rows.extend(save_selected_small_images(selected_results, label_value))
    panel_stem = f'gradcam_label{label_value}_panel'
    panel_pdf, panel_png = make_label_panel(selected_results, label_value, panel_stem)
    print('Saved panel for label', label_value, ':', panel_pdf, panel_png)

selected_df = pd.DataFrame(selected_rows)
selected_path = GRADCAM_OUT_DIR / 'gradcam_epoch20_selected_examples.xlsx'
selected_df.to_excel(selected_path, index=False)
print('Saved selected examples:', selected_path)
print('Expected small-image pairs:', N_EXAMPLES_PER_LABEL * 2 * 2)
print('Output folder:', GRADCAM_OUT_DIR)
selected_df[[
    'label_group', 'example_order', 'patient_set', 'patient_index', 'display_z_index',
    'probability_class1', 'hot_in_tumor_fraction_slice', 'tumor_cam_fraction_volume',
    'tumor_area_fraction_on_display_slice', 'tumor_fully_in_view_on_display_slice',
    'tumor_touches_view_border_on_display_slice', 'localization_score'
]]

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Model: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-20.pt
Prediction: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/predictions/prediction_eval_epoch20_fold0123456.xlsx
Output folder: /host/d/projects/Habitats/results/grad-cam
Epoch: 20
Best threshold: 0.3803069591522217
Prediction table shape: (348, 5)



Scanning Grad-CAM candidates for label = 1


  Failed: set_1 91 RuntimeError('No eligible display slice with tumor mask fully inside the 96x96 view. This case is skipped for representative Grad-CAM selection.')


  Failed: set_2 83 RuntimeError('No eligible display slice with tumor mask fully inside the 96x96 view. This case is skipped for representative Grad-CAM selection.')


...

  scanned 250 cases for label 0


Display-quality eligible cases for label 0 : 252 / 252
Selected cases for label 0
  1. set_2/80 z=44 prob=0.271 hot_in_tumor=0.919 tumor_cam=0.757 area_frac=0.476 touch_border=False score=3.159
  2. set_2/251 z=13 prob=0.358 hot_in_tumor=0.989 tumor_cam=0.528 area_frac=0.509 touch_border=False score=3.129
  3. set_2/220 z=46 prob=0.361 hot_in_tumor=0.977 tumor_cam=0.560 area_frac=0.543 touch_border=False score=3.036
  4. set_1/96 z=39 prob=0.180 hot_in_tumor=1.000 tumor_cam=0.375 area_frac=0.563 touch_border=False score=2.985
  5. set_2/44 z=44 prob=0.064 hot_in_tumor=1.000 tumor_cam=0.464 area_frac=0.517 touch_border=False score=2.981

Saved candidate ranking: /host/d/projects/Habitats/results/grad-cam/gradcam_epoch20_candidate_ranking.xlsx


Saved panel for label 1 : /host/d/projects/Habitats/results/grad-cam/gradcam_label1_panel.pdf /host/d/projects/Habitats/results/grad-cam/gradcam_label1_panel.png


Saved panel for label 0 : /host/d/projects/Habitats/results/grad-cam/gradcam_label0_panel.pdf /host/d/projects/Habitats/results/grad-cam/gradcam_label0_panel.png
Saved selected examples: /host/d/projects/Habitats/results/grad-cam/gradcam_epoch20_selected_examples.xlsx
Expected small-image pairs: 20
Output folder: /host/d/projects/Habitats/results/grad-cam


,label_group,example_order,patient_set,patient_index,display_z_index,probability_class1,hot_in_tumor_fraction_slice,tumor_cam_fraction_volume,tumor_area_fraction_on_display_slice,tumor_fully_in_view_on_display_slice,tumor_touches_view_border_on_display_slice,localization_score
0,1,1,set_1,147,27,0.863931,0.986985,0.303450,0.418511,True,False,2.793824
1,1,2,set_2,84,27,0.945033,0.989696,0.267778,0.523003,True,False,2.753648
2,1,3,set_2,11,27,0.730959,1.000000,0.309379,0.630859,True,False,2.747562
3,1,4,set_2,58,26,0.979185,0.988612,0.199972,0.472656,True,False,2.721607
4,1,5,set_2,146,27,0.918914,0.987527,0.240136,0.365885,True,False,2.711137
5,0,1,set_2,80,44,0.271126,0.919197,0.757469,0.475803,True,False,3.158984
6,0,2,set_2,251,13,0.358291,0.989154,0.527725,0.509223,True,False,3.128729
7,0,3,set_2,220,46,0.361475,0.976681,0.560040,0.543186,True,False,3.036174
8,0,4,set_1,96,39,0.180221,1.000000,0.375417,0.562717,True,False,2.984543
9,0,5,set_2,44,44,0.063632,1.000000,0.464208,0.516819,True,False,2.981241


# 3D ResNet Grad-CAM

Generate representative Grad-CAM figures for the selected 3D ResNet model.

In [2]:

# ============================================================
# Section 1. Imports and settings
# ============================================================

import os
import sys
from pathlib import Path

sys.path.append('/host/d/Github/')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

import Osteosarcoma.Image_3D.Generator_ResNet as Generator_ResNet
import Osteosarcoma.Image_3D.resnet.model as resnet_model

# ------------------------------
# Font / figure settings
# ------------------------------
font_candidates = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf',
]
for font_path in font_candidates:
    if os.path.exists(font_path):
        font_manager.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'Times New Roman'
        break

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 0.8

# ------------------------------
# Paths and model settings
# ------------------------------
LABEL = 'Prognosis'
TRIAL_NAME = 'resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam'
MODEL_SETTING = 'random0_all_fold56'
EPOCH = 13
MODEL_DEPTH = 18
IN_CHANNELS = 3
TARGET_SIZE = (96, 96, 64)
DATA_ROOT = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'
MODEL_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/models/model-{EPOCH}.pt'
PREDICTION_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/predictions/prediction_eval_epoch{EPOCH}_fold0123456.xlsx'
METRICS_PATH = f'/host/d/projects/Habitats/models/{LABEL}/{TRIAL_NAME}/{MODEL_SETTING}/predictions/metrics_eval_epoch{EPOCH}_fold0123456.xlsx'
RESULTS_DIR = Path('/host/d/projects/Habitats/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# User-selected representative cases.
REPRESENTATIVE_CASES = [
    {
        'risk_name': 'positive',
        'patient_set': 'set_1',
        'patient_index': '91',
        'label': 1,
        'target_class': 1,
        'output_path': RESULTS_DIR / 'gradcam_3d_positive_set_1_91.pdf',
    },
    {
        'risk_name': 'negative',
        'patient_set': 'set_1',
        'patient_index': '145',
        'label': 0,
        'target_class': 0,
        'output_path': RESULTS_DIR / 'gradcam_3d_negative_set_1_145.pdf',
    },
]

print('Device:', DEVICE)
print('Model:', MODEL_PATH)
print('Prediction:', PREDICTION_PATH)


Device: cuda
Model: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/models/model-13.pt
Prediction: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold56/predictions/prediction_eval_epoch13_fold0123456.xlsx


In [3]:

# ============================================================
# Section 2. Helper functions
# ============================================================

class GradCAM3D:
    def __init__(self, model, target_layer):
        self.activations = []
        self.gradients = []
        self.handles = [
            target_layer.register_forward_hook(self._forward_hook),
            target_layer.register_full_backward_hook(self._backward_hook),
        ]

    def _forward_hook(self, module, inputs, output):
        self.activations.append(output)

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients.append(grad_output[0])

    def clear(self):
        self.activations = []
        self.gradients = []

    def close(self):
        for handle in self.handles:
            handle.remove()

    def compute(self, target_size):
        if len(self.activations) == 0 or len(self.gradients) == 0:
            raise RuntimeError('No activations/gradients captured. Did backward run?')
        activation = self.activations[-1]
        gradient = self.gradients[-1]
        weights = gradient.mean(dim=(2, 3, 4), keepdim=True)
        cam = (weights * activation).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=target_size, mode='trilinear', align_corners=False)
        cam = cam[0, 0].detach().cpu().numpy().astype(np.float32)
        cam_min, cam_max = float(np.min(cam)), float(np.max(cam))
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam, dtype=np.float32)
        return cam


def load_resnet_checkpoint(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            key = key[len('module.'):]
        cleaned[key] = value
    model.load_state_dict(cleaned, strict=True)
    return model


def build_model():
    model = resnet_model.build_resnet3d_model(
        model_depth=MODEL_DEPTH,
        num_classes=2,
        in_channels=IN_CHANNELS,
    )
    model = load_resnet_checkpoint(model, MODEL_PATH, DEVICE)
    model.to(DEVICE)
    model.eval()
    return model


def build_single_case_dataset(patient_set, patient_index, label):
    x_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'img.nii.gz')
    return Generator_ResNet.Dataset_3D(
        [patient_set],
        [str(patient_index)],
        [x_file],
        [int(label)],
        DATA_ROOT,
        target_image_size=TARGET_SIZE,
        normalize_factor='medicalnet',
        only_tumor_pixels='seg',
        augment_context='full',
        shuffle=False,
        augment=False,
        augment_frequency=0,
    )


def get_case_arrays(dataset):
    patient_set = dataset.patient_set_list[0]
    patient_index = dataset.patient_index_list[0]
    img_file = dataset.x_file_list[0]
    label_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'label.nii.gz')
    bbox_file = os.path.join(DATA_ROOT, patient_set, str(patient_index), 'bbox_mask.nii.gz')
    raw_stack = dataset.load_file(img_file, label_file, bbox_file)
    x_tensor, y_tensor = dataset[0]
    return raw_stack, x_tensor.unsqueeze(0), int(y_tensor.item())


def choose_largest_tumor_slice(raw_stack):
    tumor_mask = raw_stack[2] != 0
    if not np.any(tumor_mask):
        tumor_mask = raw_stack[1] != 0
    z_area = tumor_mask.sum(axis=(0, 1))
    z_index = int(np.argmax(z_area))
    return z_index, tumor_mask


def normalize_display(img):
    img = img.astype(np.float32)
    valid = img[img != 0]
    if valid.size > 20:
        lo, hi = np.percentile(valid, [1, 99])
    else:
        lo, hi = float(np.min(img)), float(np.max(img))
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((img - lo) / (hi - lo), 0, 1)


def orient_slice(slice_2d):
    return np.flipud(np.rot90(slice_2d, k=1))


def overlay_mask_on_gray(gray, mask, color=(1.0, 0.0, 0.0), alpha=0.42):
    rgb = np.repeat(gray[..., None], 3, axis=-1)
    color_arr = np.asarray(color, dtype=np.float32)
    mask = mask.astype(bool)
    rgb[mask] = (1 - alpha) * rgb[mask] + alpha * color_arr
    return np.clip(rgb, 0, 1)


def make_cam_overlay(gray, cam, alpha=0.48):
    heat_rgb = plt.get_cmap('jet')(cam)[..., :3]
    gray_rgb = np.repeat(gray[..., None], 3, axis=-1)
    return np.clip((1 - alpha) * gray_rgb + alpha * heat_rgb, 0, 1)


def cam_energy_stats(cam_volume, tumor_mask, bbox_mask):
    total = float(cam_volume.sum())
    if total <= 0:
        return np.nan, np.nan
    tumor_fraction = float(cam_volume[tumor_mask].sum() / total) if np.any(tumor_mask) else np.nan
    bbox_fraction = float(cam_volume[bbox_mask].sum() / total) if np.any(bbox_mask) else np.nan
    return tumor_fraction, bbox_fraction


def run_gradcam_for_case(model, gradcam, case_info):
    dataset = build_single_case_dataset(case_info['patient_set'], case_info['patient_index'], case_info['label'])
    raw_stack, x_tensor, y_true = get_case_arrays(dataset)
    x_tensor = x_tensor.to(DEVICE, dtype=torch.float32)

    model.eval()
    model.zero_grad(set_to_none=True)
    gradcam.clear()

    logits = model(x_tensor)
    prob = torch.softmax(logits, dim=1)[0, 1].detach().cpu().item()
    pred_class = int(torch.argmax(logits, dim=1).detach().cpu().item())
    target_class = int(case_info['target_class'])
    logits[0, target_class].backward()
    cam_volume = gradcam.compute(TARGET_SIZE)

    largest_z_index, tumor_mask_volume = choose_largest_tumor_slice(raw_stack)
    bbox_mask_volume = raw_stack[1] != 0
    tumor_cam_fraction, bbox_cam_fraction = cam_energy_stats(cam_volume, tumor_mask_volume, bbox_mask_volume)

    # For visualization, use the tumor-containing slice where Grad-CAM overlaps
    # the tumor most strongly. This avoids showing a large-tumor slice where the
    # heatmap itself is weak or displaced.
    z_tumor_area = tumor_mask_volume.sum(axis=(0, 1))
    z_cam_tumor_overlap = (cam_volume * tumor_mask_volume).sum(axis=(0, 1))
    min_area = max(5, 0.10 * float(z_tumor_area.max()))
    valid_z = np.where(z_tumor_area >= min_area)[0]
    if valid_z.size > 0:
        z_index = int(valid_z[np.argmax(z_cam_tumor_overlap[valid_z])])
    else:
        z_index = int(largest_z_index)

    mri_slice = orient_slice(raw_stack[0, :, :, z_index])
    tumor_mask_slice = orient_slice(tumor_mask_volume[:, :, z_index])
    cam_slice = orient_slice(cam_volume[:, :, z_index])
    gray = normalize_display(mri_slice)
    tumor_overlay = overlay_mask_on_gray(gray, tumor_mask_slice, color=(1.0, 0.0, 0.0), alpha=0.42)
    cam_overlay = make_cam_overlay(gray, cam_slice, alpha=0.48)

    return {
        'z_index': z_index,
        'probability': float(prob),
        'prediction_class': pred_class,
        'true_label': y_true,
        'target_class': target_class,
        'gray': gray,
        'tumor_mask_slice': tumor_mask_slice,
        'tumor_overlay': tumor_overlay,
        'cam_slice': cam_slice,
        'cam_overlay': cam_overlay,
        'tumor_cam_fraction': tumor_cam_fraction,
        'bbox_cam_fraction': bbox_cam_fraction,
        'tumor_area_on_display_slice': int(np.sum(tumor_mask_slice)),
    }


def plot_case(result, output_path):
    fig = plt.figure(figsize=(7.2, 2.35), dpi=300)
    gs = fig.add_gridspec(
        1,
        4,
        width_ratios=[1, 1, 1, 0.055],
        left=0.01,
        right=0.96,
        top=0.99,
        bottom=0.01,
        wspace=0.03,
    )
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[0, 2])
    cax = fig.add_subplot(gs[0, 3])

    ax0.imshow(result['tumor_overlay'], vmin=0, vmax=1)
    im = ax1.imshow(result['cam_slice'], cmap='jet', vmin=0, vmax=1)
    ax2.imshow(result['cam_overlay'], vmin=0, vmax=1)

    for ax in [ax0, ax1, ax2]:
        ax.set_axis_off()

    cbar = fig.colorbar(im, cax=cax)
    cbar.set_ticks(np.arange(0, 1.01, 0.2))
    cbar.ax.tick_params(labelsize=7, length=2, width=0.6)

    save_pdf_and_ppt_safe_svg(fig, output_path, bbox_inches='tight', pad_inches=0.01)
    plt.close(fig)


In [4]:

# ============================================================
# Section 3. Generate Grad-CAM figures
# ============================================================

prediction_df = pd.read_excel(PREDICTION_PATH)
metrics_df = pd.read_excel(METRICS_PATH)
best_threshold = float(metrics_df['Best_threshold'].iloc[0])

print('Best threshold:', best_threshold)
print('Selected representative cases:')
for case_info in REPRESENTATIVE_CASES:
    row = prediction_df[
        (prediction_df['Patient_set'].astype(str) == case_info['patient_set'])
        & (prediction_df['Patient_index'].astype(str) == str(case_info['patient_index']))
    ]
    if row.shape[0] != 1:
        raise RuntimeError(f'Could not uniquely find prediction row for {case_info}')
    prob = float(row['probability'].iloc[0])
    label = int(row['label'].iloc[0])
    fold = int(row['fold'].iloc[0])
    pred = int(prob >= best_threshold)
    case_info['fold'] = fold
    case_info['label'] = label
    print(case_info['risk_name'], case_info['patient_set'], case_info['patient_index'], 'fold', fold, 'label', label, 'prob', prob, 'pred@thr', pred)

model = build_model()
gradcam = GradCAM3D(model, model.layer4[-1])

summary_rows = []
for case_info in REPRESENTATIVE_CASES:
    print('\nRunning Grad-CAM:', case_info['risk_name'], case_info['patient_set'], case_info['patient_index'])
    result = run_gradcam_for_case(model, gradcam, case_info)
    plot_case(result, case_info['output_path'])

    summary_rows.append({
        'risk_name': case_info['risk_name'],
        'patient_set': case_info['patient_set'],
        'patient_index': case_info['patient_index'],
        'fold': case_info['fold'],
        'true_label': result['true_label'],
        'target_class_for_gradcam': result['target_class'],
        'probability_class1': result['probability'],
        'prediction_class_argmax': result['prediction_class'],
        'display_z_index': result['z_index'],
        'tumor_area_on_display_slice': result['tumor_area_on_display_slice'],
        'tumor_cam_fraction_volume': result['tumor_cam_fraction'],
        'bbox_cam_fraction_volume': result['bbox_cam_fraction'],
        'output_path': str(case_info['output_path']),
    })
    print('  probability:', result['probability'])
    print('  prediction class:', result['prediction_class'])
    print('  display z index:', result['z_index'])
    print('  tumor CAM fraction:', result['tumor_cam_fraction'])
    print('  bbox CAM fraction:', result['bbox_cam_fraction'])
    print('  saved:', case_info['output_path'])

gradcam.close()
summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / 'gradcam_3d_representative_cases.xlsx'
summary_df.to_excel(summary_path, index=False)
print('\nSaved summary:', summary_path)
summary_df


Best threshold: 0.07977509498596191
Selected representative cases:
positive set_1 91 fold 1 label 1 prob 0.578704833984375 pred@thr 1
negative set_1 145 fold 4 label 0 prob 0.007211805786937475 pred@thr 0



Running Grad-CAM: positive set_1 91


  probability: 0.578704833984375
  prediction class: 1
  display z index: 58
  tumor CAM fraction: 0.7370546211317885
  bbox CAM fraction: 0.9578974932310411
  saved: /host/d/projects/Habitats/results/gradcam_3d_positive_set_1_91.pdf

Running Grad-CAM: negative set_1 145


  probability: 0.007211805786937475
  prediction class: 0
  display z index: 41
  tumor CAM fraction: 0.09628366537514223
  bbox CAM fraction: 0.261496619129551
  saved: /host/d/projects/Habitats/results/gradcam_3d_negative_set_1_145.pdf

Saved summary: /host/d/projects/Habitats/results/gradcam_3d_representative_cases.xlsx


,risk_name,patient_set,patient_index,fold,true_label,target_class_for_gradcam,probability_class1,prediction_class_argmax,display_z_index,tumor_area_on_display_slice,tumor_cam_fraction_volume,bbox_cam_fraction_volume,output_path
0,positive,set_1,91,1,1,1,0.578705,1,58,6301,0.737055,0.957897,/host/d/projects/Habitats/results/gradcam_3d_p...
1,negative,set_1,145,4,0,0,0.007212,0,41,1838,0.096284,0.261497,/host/d/projects/Habitats/results/gradcam_3d_n...
